# UNet и Семантическая Сегментация

В этом ноутбуке мы разберём **UNet** — одну из самых популярных архитектур для семантической сегментации изображений.

**Что ты изучишь:**
1. Архитектура UNet (encoder-decoder + skip connections)
2. Построение UNet с нуля
3. Улучшенные версии (ResUNet, Attention UNet)
4. Обучение на датасете сегментации
5. Метрики: IoU и Dice
6. Визуализация предсказаний

> **Запускай в Colab** — Runtime → T4 GPU.

## Зачем нужна семантическая сегментация?

В отличие от классификации (одна метка на изображение) и детекции (рамки + классы), **сегментация** даёт точную маску для каждого пикселя.

Применения:
- Медицина (выделение опухолей, органов)
- Автономное вождение (разметка дороги, пешеходов)
- Спутниковые снимки (выделение зданий, лесов, воды)
- Промышленность (дефекты на производстве)

# 1. Архитектура UNet

UNet состоит из двух частей:

1. **Encoder** (сжимающий путь) — извлекает признаки
2. **Decoder** (расширяющий путь) — восстанавливает пространственное разрешение

Ключевой элемент — **skip connections** (передача признаков с encoder на decoder на каждом уровне).

# 2. Реализация UNet с нуля

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, n_classes=1):
        super().__init__()
        self.encoder1 = DoubleConv(in_channels, 64)
        self.encoder2 = DoubleConv(64, 128)
        self.encoder3 = DoubleConv(128, 256)
        self.encoder4 = DoubleConv(256, 512)
        
        self.pool = nn.MaxPool2d(2)
        
        self.bottleneck = DoubleConv(512, 1024)
        
        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.decoder4 = DoubleConv(1024, 512)
        
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.decoder3 = DoubleConv(512, 256)
        
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.decoder2 = DoubleConv(256, 128)
        
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.decoder1 = DoubleConv(128, 64)
        
        self.final = nn.Conv2d(64, n_classes, 1)

    def forward(self, x):
        # Encoder
        e1 = self.encoder1(x)
        e2 = self.encoder2(self.pool(e1))
        e3 = self.encoder3(self.pool(e2))
        e4 = self.encoder4(self.pool(e3))
        
        b = self.bottleneck(self.pool(e4))
        
        # Decoder
        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.decoder4(d4)
        
        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.decoder3(d3)
        
        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.decoder2(d2)
        
        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.decoder1(d1)
        
        return self.final(d1)

# 3. Улучшенные версии UNet

### ResUNet (с residual блоками)

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = DoubleConv(in_ch, out_ch)
        self.shortcut = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    
    def forward(self, x):
        return self.conv(x) + self.shortcut(x)

class ResUNet(nn.Module):
    def __init__(self, in_channels=3, n_classes=1):
        super().__init__()
        # ... (аналогично UNet, но с ResBlock вместо DoubleConv)
        pass

### Attention UNet

In [ ]:
class AttentionBlock(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, 1),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, 1),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, 1),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        
    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.psi(F.relu(g1 + x1))
        return x * psi

# 4. Обучение UNet

In [ ]:
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader

# Пример: Oxford Pets (сегментация)
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

trainset = torchvision.datasets.OxfordIIITPet(
    root='./data', split='trainval', target_types='segmentation',
    download=True, transform=transform
)
trainloader = DataLoader(trainset, batch_size=8, shuffle=True)

In [ ]:
model = UNet(in_channels=3, n_classes=1).cuda()
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(10):
    for images, masks in trainloader:
        images, masks = images.cuda(), masks.cuda().float()
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
    
    print(f'Epoch {epoch+1}, Loss: {loss.item():.4f}')

# 5. Метрики: IoU и Dice

In [ ]:
def iou_score(outputs, targets, threshold=0.5):
    outputs = (outputs > threshold).float()
    intersection = (outputs * targets).sum()
    union = outputs.sum() + targets.sum() - intersection
    return (intersection + 1e-6) / (union + 1e-6)

def dice_score(outputs, targets, threshold=0.5):
    outputs = (outputs > threshold).float()
    intersection = (outputs * targets).sum()
    return (2 * intersection + 1e-6) / (outputs.sum() + targets.sum() + 1e-6)

# 6. Упражнения

### Упражнение 1: Реализуй ResUNet полностью

In [ ]:
# Твой код здесь

### Упражнение 2: Добавь Attention в UNet

In [ ]:
# Твой код здесь

### Упражнение 3: Сегментация на своём датасете

Возьми любой датасет с масками (например, через `torchvision.datasets`) и обучи UNet. Визуализируй результаты.

---

**Готово!**  
Ты освоил одну из самых важных архитектур в компьютерном зрении.

Следующий шаг — **WS7: YOLO и Object Detection**.